In [129]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [130]:
df = sns.load_dataset('titanic')

In [131]:
df

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True


In [132]:
df.isna().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [133]:
df['family'] = df['sibsp'] + df['parch']

In [134]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

In [135]:
df.columns

Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town',
       'alive', 'alone', 'family'],
      dtype='object')

In [136]:
x=df[['pclass','sex','age','fare','adult_male','embark_town']]
y=df['survived']

In [137]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,shuffle=True)

In [138]:
x_train

,pclass,sex,age,fare,adult_male,embark_town
331,1,male,45.5,28.5000,True,Southampton
733,2,male,23.0,13.0000,True,Southampton
382,3,male,32.0,7.9250,True,Southampton
704,3,male,26.0,7.8542,True,Southampton
813,3,female,6.0,31.2750,False,Southampton
...,...,...,...,...,...,...
106,3,female,21.0,7.6500,False,Southampton
270,1,male,NaN,31.0000,True,Southampton
860,3,male,41.0,14.1083,True,Southampton
435,1,female,14.0,120.0000,False,Southampton


In [139]:
y_train

331    0
733    0
382    0
704    0
813    0
      ..
106    1
270    0
860    0
435    1
102    0
Name: survived, Length: 712, dtype: int64

In [140]:
x_test

,pclass,sex,age,fare,adult_male,embark_town
709,3,male,NaN,15.2458,True,Cherbourg
439,2,male,31.0,10.5000,True,Southampton
840,3,male,20.0,7.9250,True,Southampton
720,2,female,6.0,33.0000,False,Southampton
39,3,female,14.0,11.2417,False,Cherbourg
...,...,...,...,...,...,...
433,3,male,17.0,7.1250,True,Southampton
773,3,male,NaN,7.2250,True,Cherbourg
25,3,female,38.0,31.3875,False,Southampton
84,2,female,17.0,10.5000,False,Southampton


In [141]:
y_test

709    1
439    0
840    0
720    1
39     1
      ..
433    0
773    0
25     1
84     1
10     1
Name: survived, Length: 179, dtype: int64

In [142]:
x_train

,pclass,sex,age,fare,adult_male,embark_town
331,1,male,45.5,28.5000,True,Southampton
733,2,male,23.0,13.0000,True,Southampton
382,3,male,32.0,7.9250,True,Southampton
704,3,male,26.0,7.8542,True,Southampton
813,3,female,6.0,31.2750,False,Southampton
...,...,...,...,...,...,...
106,3,female,21.0,7.6500,False,Southampton
270,1,male,NaN,31.0000,True,Southampton
860,3,male,41.0,14.1083,True,Southampton
435,1,female,14.0,120.0000,False,Southampton


In [143]:
categorical_col = ['sex','adult_male','embark_town']
numerical_col = ['pclass','age','fare']

In [144]:
numerical_transform = Pipeline(steps=[
    ('knnimput',KNNImputer(n_neighbors=3)),
    ('scaler',StandardScaler())
])

In [145]:
categorical_transform = Pipeline(steps=[
    ('mode_handling',SimpleImputer(strategy='most_frequent')),
    ('one',OneHotEncoder(drop='first'))
])

In [146]:
processor = ColumnTransformer([
    ('num',numerical_transform,numerical_col),
    ('cate',categorical_transform,categorical_col)
]
)

In [ ]:
model_pipline = Pipeline(steps=[
    ('process',processor),
    # ('model',LogisticRegression(max_iter=100)) using this model accuracy_score is 0.8044692737430168 so i choose this model.
    # ('model',DecisionTreeClassifier()) using this model accuracy_score is 0.7653631284916201
])

In [148]:
model_pipline.fit(x_train,y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('process', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cate', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contai

In [149]:
y_pred = model_pipline.predict(x_test)

In [150]:
y_pred

array([0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       1, 1, 1])

In [151]:
from sklearn.metrics import accuracy_score

In [152]:
print(f'accuracy_score is this model using knn_imputer{accuracy_score(y_test,y_pred)}')

accuracy_score is this model using knn_imputer0.8044692737430168


In [153]:
from sklearn.metrics import classification_report

In [154]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.84      0.83      0.83       105
           1       0.76      0.77      0.77        74

    accuracy                           0.80       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.80      0.80      0.80       179



In [155]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred))

[[87 18]
 [17 57]]
